In [4]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), '../'))

import math
import optuna
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import linregress
from datetime import datetime

from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

import lightgbm as lgb
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, precision_score, average_precision_score

## Crawl data

In [5]:
from vnstock import Vnstock
from vnstock.explorer.vci.listing import Listing

symbols_by_industries = Listing().symbols_by_industries()
symbols_by_exchange = Listing().symbols_by_exchange()
symbols_info = symbols_by_industries.merge(symbols_by_exchange)

Phiên bản Vnstock 3.2.2 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnstock --upgrade`.
Lịch sử phiên bản: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban
Phiên bản hiện tại 3.1.0.2

In [6]:
symbols_list = symbols_info['symbol'].tolist()
data_count_save_path = '/remote/vast0/share-mv/tien/project/vnstock/data/data_count.csv'

data_count = None
keep_symbols = None
if os.path.exists(data_count_save_path):
    data_count = pd.read_csv(data_count_save_path)

In [7]:
keep_symbols = []
data_count = []
all_data = {}
for symbol in tqdm(symbols_list):
    save_path = f'/remote/vast0/share-mv/tien/project/vnstock/data/stocks/{symbol}.csv'
    if os.path.exists(save_path):
        df = pd.read_csv(save_path)
    else:
        try:
            vn_stock = Vnstock()
            stock = vn_stock.stock(symbol=symbol, source='VCI')
            df = stock.quote.history(start='2020-01-01', end='2025-02-25')
            df.to_csv(f'/remote/vast0/share-mv/tien/project/vnstock/data/stocks/{symbol}.csv', index=False)
        except Exception as e:
            print(e)
            continue
    if ((df['volume']*df['close']).rolling(10).mean()).values[-1].tolist() > 1e6 and (df['volume'] == 0).sum() < 10 and df.shape[0] > 800:
        data_count.append([symbol, df.shape[0]])
        all_data[symbol] = df
        keep_symbols.append(symbol)
    

 38%|███▊      | 608/1591 [00:02<00:06, 140.79it/s]

Không tìm thấy dữ liệu. Vui lòng kiểm tra lại mã chứng khoán hoặc thời gian truy xuất.


 62%|██████▏   | 991/1591 [00:03<00:03, 193.03it/s]

Expecting value: line 1 column 1 (char 0)


2025-03-25 11:04:02 - vnstock.common.data.data_explorer - WARNING - Thông tin niêm yết & giao dịch sẽ được truy xuất từ TCBS
 67%|██████▋   | 1067/1591 [00:04<00:04, 111.97it/s]

Không tìm thấy dữ liệu. Vui lòng kiểm tra lại mã chứng khoán hoặc thời gian truy xuất.


100%|██████████| 1591/1591 [00:05<00:00, 277.35it/s]


In [8]:
if data_count is None:
    data_count = pd.DataFrame(data_count, columns=['symbol', 'count'])
    data_count.to_csv(data_count_save_path, index=False)

## Features Engineering

In [9]:
len(keep_symbols)

253

In [10]:
print(keep_symbols)

['ACB', 'AGR', 'AAA', 'BNA', 'AGG', 'CSC', 'CTF', 'DRI', 'DHG', 'DGC', 'DLG', 'BWE', 'DPM', 'BID', 'ACV', 'ELC', 'AMV', 'DPR', 'EVG', 'ANV', 'CSV', 'APG', 'APH', 'ICT', 'FCN', 'CSM', 'FMC', 'FIT', 'DHA', 'DCM', 'FPT', 'BSI', 'BMI', 'DVP', 'BVB', 'CTS', 'BMP', 'BCG', 'BOT', 'HAH', 'CNG', 'AAS', 'C69', 'D2D', 'BFC', 'LTG', 'CII', 'HBC', 'HAX', 'AAV', 'ABB', 'DRC', 'ASM', 'ABI', 'CTG', 'CST', 'CMG', 'LSS', 'CTI', 'DRH', 'FIR', 'EVF', 'BCM', 'DBD', 'GEX', 'HPG', 'HQC', 'CCL', 'IDI', 'DXG', 'HHP', 'TCH', 'HT1', 'CTR', 'BVH', 'HTN', 'DXS', 'IDJ', 'DIG', 'TNH', 'SSH', 'BVS', 'DPG', 'IJC', 'PHP', 'DBC', 'CEO', 'HHS', 'EIB', 'ITD', 'HCM', 'BMC', 'HDC', 'HUT', 'HDG', 'C4G', 'KDC', 'KDH', 'KHG', 'GAS', 'KHP', 'JVC', 'GEG', 'GMD', 'KSB', 'HVH', 'HAP', 'SGP', 'KVC', 'DHC', 'LAS', 'LCG', 'IMP', 'LHG', 'KOS', 'LIG', 'DHT', 'KBC', 'MBB', 'DGW', 'LPB', 'GIL', 'DST', 'ITA', 'FTS', 'MSN', 'MSR', 'L14', 'NAG', 'CTD', 'MWG', 'NRC', 'HNG', 'NLG', 'BSR', 'HDB', 'PSH', 'NT2', 'NTL', 'NTC', 'OCB', 'MCH', 'DCL'

In [11]:
def create_candle_related_features(df):
    df['prev_max_high_5d'] = df['high'].rolling(window=5).max().shift(1)
    df['prev_min_low_5d'] = df['low'].rolling(window=5).min().shift(1)
    df['prev_high'] = df['high'].shift(1)
    df['prev_low'] = df['low'].shift(1)

    for price_type in ['close', 'open', 'high', 'low']:
        df[f'cur_{price_type}_vs_prev_max_high_5d_diff'] = (df[price_type] - df['prev_max_high_5d']) / df['prev_max_high_5d']
        df[f'cur_{price_type}_vs_prev_min_low_5d_diff'] = (df[price_type] - df['prev_min_low_5d']) / df['prev_min_low_5d']
        df[f'cur_{price_type}_vs_prev_max_high_diff'] = (df[price_type] - df['prev_high']) / df['prev_high']
        df[f'cur_{price_type}_vs_prev_min_low_diff'] = (df[price_type] - df['prev_low']) / df['prev_low']

    for n in [3, 5, 10, 20]:
        df[f'incr_candles_count_{n}d'] = (df['close'] > df['open']).rolling(window=n).sum()
        df[f'decr_candles_count_{n}d'] = (df['close'] < df['open']).rolling(window=n).sum()
        # Sum of absolute differences for increasing candles
        df[f'incr_candles_abs_diff_sum_{n}d'] = (
            (df['close'] / df['open'] - 1).where(df['close'] > df['open'], 0).rolling(window=n).sum()
        )
        
        # Sum of absolute differences for decreasing candles
        df[f'decr_candles_abs_diff_sum_{n}d'] = (
            (df['open'] / df['close'] - 1).where(df['close'] < df['open'], 0).rolling(window=n).sum()
        )

    del df['prev_max_high_5d'], df['prev_min_low_5d'], df['prev_high'], df['prev_low']
    return df

def feature_engineering(df):
    from talipp.ohlcv import OHLCVFactory
    from talipp.indicators import ADX, AO, Aroon, ATR, BOP, CHOP, CoppockCurve, DPO, EMV, IBS, \
        MACD, MassIndex, ROC, RSI, STC, Stoch, StochRSI, SuperTrend, TRIX, TSI, TTM, UO, VTX

    def get_volume_level(x):
        if x < 1e7:
            return 0
        if x < 5e7:
            return 1
        if x < 1e8:
            return 2
        return 3

    def incr_rate_estimate(x):
        d = len(x)
        lr = linregress(range(d), x/x.tolist()[-1])
        i = lr.intercept.tolist()
        s = lr.slope.tolist()
        if i == 0:
            return s*d
        return s*d/i

    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values(by='time')
    df['close_open_diff'] = (df['close'] - df['open']) / df['open']
    df['high_low_diff'] = (df['high'] - df['low']) / df['low']
    df['close_low_diff'] = (df['close'] - df['low']) / df['low']
    df['volume_level'] = (df['volume'] * df['close']).apply(get_volume_level)
    df['volume_level'] = df['volume_level'].astype("category")

    # Percentage change
    for col in ['close', 'volume', 'high', 'low']:
        for d in [1, 3, 5, 10, 20, 40]:
            new_col = f'{col}_{d}d_pct_change'
            df[new_col] = df[col].pct_change(d)
            df.loc[df[new_col] == np.inf, new_col] = 1
            df.loc[df[new_col] == -np.inf, new_col] = -1

        for date_suffix in ['3d_vs_5d_pct_change', '5d_vs_10d_pct_change', '10d_vs_20d_pct_change']:
            new_col = f'{col}_{date_suffix}'
            first_date = int(date_suffix.split('_')[0][:-1])
            second_date = int(date_suffix.split('_')[2][:-1])
            df[new_col] = df[col].shift(first_date)/df[col].shift(second_date)-1
            df.loc[df[new_col] == np.inf, new_col] = 1
            df.loc[df[new_col] == -np.inf, new_col] = -1

    for col in ['close', 'volume', 'high', 'low']:
        for d in [5, 10, 20, 40]:
            df[f'{col}_avg{d}'] = df[col].rolling(window=d).mean()
            df[f'cur_{col}_vs_avg{d}_diff'] = (df[col] - df[f'{col}_avg{d}']) / df[f'{col}_avg{d}']
            del df[f'{col}_avg{d}']

    for col in ['volume', 'close', 'high', 'low']:
        for d in [5, 10, 20, 40]:
            df[f'{col}_slope{d}'] = df[col].rolling(d).apply(incr_rate_estimate)

    for col in ['volume', 'close', 'high', 'low']:
        df[f'{col}_slope5_vs_slope10_diff'] = (1+df[f'{col}_slope5']) / (1+df[f'{col}_slope10'])-1
        df[f'{col}_slope5_vs_slope20_diff'] = (1+df[f'{col}_slope5']) / (1+df[f'{col}_slope20'])-1
        df[f'{col}_slope10_vs_slope20_diff'] = (1+df[f'{col}_slope10']) / (1+df[f'{col}_slope20'])-1
        df[f'{col}_slope10_vs_slope40_diff'] = (1+df[f'{col}_slope10']) / (1+df[f'{col}_slope40'])-1
        df[f'{col}_slope20_vs_slope40_diff'] = (1+df[f'{col}_slope20']) / (1+df[f'{col}_slope40'])-1

    df = create_candle_related_features(df)
    ohlcv = OHLCVFactory.from_dict({
        "open": df['open'],
        "high": df['high'],
        "low": df['low'],
        "close": df['close'],
        "volume": df['volume']
    })
    close = df['close']
    adx = ADX(10, 10, ohlcv)
    adx_plus_di = [adx_val.plus_di if adx_val is not None else None for adx_val in adx.output_values]
    adx_minus_di = [adx_val.minus_di if adx_val is not None else None for adx_val in adx.output_values]
    ao = AO(5, 34, ohlcv).output_values
    aroon = Aroon(10, ohlcv)
    aroon_up = [a.up if a is not None else None for a in aroon.output_values]
    aroon_down = [a.down if a is not None else None for a in aroon.output_values]
    # atr = ATR(10, ohlcv).output_values
    bop = BOP(ohlcv).output_values
    chop = CHOP(10, ohlcv).output_values
    coppock_curve = CoppockCurve(11, 10, 10, close).output_values
    dpo = DPO(20, close).output_values
    emv = EMV(10, 10000, ohlcv).output_values
    ibs = IBS(ohlcv).output_values
    macd = MACD(12, 26, 9, close).output_values
    macd_hist = [m.histogram if m is not None else None for m in macd]
    macd_signal = [m.signal if m is not None else None for m in macd]
    macd_val = [m.macd if m is not None else None for m in macd]
    mass_index = MassIndex(9, 9, 10, ohlcv).output_values
    roc = ROC(9, close).output_values
    rsi = RSI(10, close).output_values
    stc = STC(23, 50, 10, 3, close).output_values
    stoch = Stoch(10, 3, ohlcv)
    stoch_k = [s.k if s is not None else None for s in stoch.output_values]
    stoch_d = [s.d if s is not None else None for s in stoch.output_values]
    stoch_rsi = StochRSI(10, 10, 3, 3, close)
    stoch_rsi_k = [s.k if s is not None else None for s in stoch_rsi.output_values]
    stoch_rsi_d = [s.d if s is not None else None for s in stoch_rsi.output_values]
    # super_trend = [s.value if s is not None else None for s in SuperTrend(10, 3, ohlcv).output_values]
    # trix = TRIX(18, close).output_values
    tsi = TSI(13, 25, close).output_values
    ttm = TTM(20, input_values = ohlcv)
    ttm_histogram = [t.histogram if t is not None else None for t in ttm.output_values]
    uo = UO(7, 10, 20, ohlcv).output_values
    vtx = VTX(10, ohlcv)
    vtx_plus = [v.plus_vtx if v is not None else None for v in vtx.output_values]
    vtx_minus = [v.minus_vtx if v is not None else None for v in vtx.output_values]
    df['adx_plus_di'] = adx_plus_di
    df['adx_minus_di'] = adx_minus_di
    df['ao'] = ao
    df['aroon_up'] = aroon_up
    df['aroon_down'] = aroon_down
    # df['atr'] = atr
    df['bop'] = bop
    df['chop'] = chop
    df['coppock_curve'] = coppock_curve
    df['dpo'] = dpo
    df['emv'] = emv
    df['ibs'] = ibs
    df['macd_hist'] = macd_hist
    df['macd_signal'] = macd_signal
    df['macd_val'] = macd_val
    df['mass_index'] = mass_index
    df['roc'] = roc
    df['rsi'] = rsi
    df['stc'] = stc
    df['stoch_k'] = stoch_k
    df['stoch_d'] = stoch_d
    df['stoch_rsi_k'] = stoch_rsi_k
    df['stoch_rsi_d'] = stoch_rsi_d
    # df['super_trend'] = super_trend
    # df['trix'] = trix
    df['tsi'] = tsi
    df['ttm_histogram'] = ttm_histogram
    df['uo'] = uo
    df['vtx_plus'] = vtx_plus
    df['vtx_minus'] = vtx_minus
    
    # shift_cols = [c for c in df.columns if not c.startswith('prev') and c not in ['time', 'open', 'high', 'low', 'close', 'volume', 'next_close', 'label', 'weight']]
    # for shift_day in range(1,4):
    #     df[[f'prev{shift_day}d_{sc}' for sc in shift_cols]] = df.shift(shift_day)[shift_cols]

    return df

## Label

In [12]:
def label(df):
    upper_rate = 1.05
    # df['label'] = df['close'].shift(-3) / df['close'] - 1
    df['next_close'] = df['close'].rolling(10).max().shift(-10)
    df['next_min_close'] = df['close'].rolling(10).min().shift(-10)

    df['label'] = df['next_close'] > df['open'].shift(-1) * upper_rate
    df['label'] &= df['next_min_close'] > df['open'].shift(-1) * 0.95

    df['label'] = df['next_close'] > df['open'].shift(-1) * upper_rate

    df['weight'] = 1+(df['next_close']/df['open'].shift(-1)-1).abs()
    
    return df

In [12]:
# old_all_df = all_df.copy()
# all_df = old_all_df.copy()

In [23]:
all_df = []
c = 0
for symbol, df in all_data.items():
    print(symbol)
    df = feature_engineering(df)
    df = label(df)
    df['symbol'] = symbol
    all_df.append(df)
    c += 1

all_df = pd.concat(all_df)
all_df = all_df.dropna()
all_df = all_df[(abs(all_df['close']/all_df['open']-1) > 0.04) | (abs(all_df['high']/all_df['low']-1) > 0.08) | (abs(all_df['cur_volume_vs_avg10_diff']) > 1.)]
# all_df = all_df[(all_df['cur_volume_vs_avg10_diff'].abs() > 0.5) | (all_df['prev1d_cur_volume_vs_avg10_diff'].abs() > 0.5) | (all_df['prev2d_cur_volume_vs_avg10_diff'].abs() > 0.5) | (all_df['prev3d_cur_volume_vs_avg10_diff'].abs() > 0.5)].reset_index(drop=True)

ACB
AGR
AAA
BNA
AGG
CSC
CTF
DRI
DHG
DGC
DLG
BWE
DPM
BID
ACV
ELC
AMV
DPR
EVG
ANV
CSV
APG
APH
ICT
FCN
CSM
FMC
FIT
DHA
DCM
FPT
BSI
BMI
DVP
BVB
CTS
BMP
BCG
BOT


VTX: ATR value is 0, but plus_vm and minus_vm are not. This should not happen, return (0,0).


HAH
CNG
AAS
C69
D2D
BFC
LTG
CII
HBC
HAX
AAV
ABB
DRC
ASM
ABI
CTG
CST
CMG
LSS
CTI
DRH
FIR
EVF
BCM


IBS: close value '22.86' must be equal to low value '23.05' when high '23.05' is equal to low '23.05', if not, the IBS value will be 1.0.


DBD
GEX
HPG
HQC
CCL
IDI
DXG
HHP
TCH
HT1
CTR
BVH
HTN
DXS
IDJ
DIG
TNH
SSH
BVS
DPG
IJC
PHP
DBC
CEO
HHS
EIB
ITD
HCM
BMC
HDC
HUT
HDG
C4G
KDC
KDH
KHG
GAS
KHP
JVC
GEG
GMD
KSB
HVH
HAP
SGP
KVC
DHC
LAS
LCG
IMP
LHG
KOS
LIG
DHT
KBC
MBB
DGW
LPB
GIL
DST
ITA
FTS
MSN
MSR
L14
NAG
CTD
MWG
NRC
HNG
NLG
BSR
HDB
PSH
NT2
NTL
NTC
OCB
MCH
DCL
MSB
PET
PLX
PC1
GKM
PNJ
PLC
PTB
PDR
DTD
PVB
LDG
PVD
PHR
IDV
POW
PVT
IDC
PVI
QTP
REE
FRT
NAF
PVP
SAB
HAG
NKG
SCG
SCS
QCG
SSB
NVL
QNS
SGN
NAB
MST
SAM
NHA
PAN
MIG


IBS: close value '6.54' must be equal to low value '6.6' when high '6.6' is equal to low '6.6', if not, the IBS value will be 1.0.
IBS: close value '5.3' must be equal to low value '5.36' when high '5.36' is equal to low '5.36', if not, the IBS value will be 1.0.


HSG
PPC
VHM
SBS
SBT
OIL
PVS
SCR
SHB
SZC
SJS
NTP
SHS
SMB
SMC
PVC
SJD
MSH
SLS
SAS


IBS: close value '20.3' must be equal to low value '19.83' when high '19.83' is equal to low '19.83', if not, the IBS value will be 1.0.


SIP
STB
SSI
SHI
TCM
TV2
TCB
TCL
TDC
THG
TIG
TIP
VIP
TNG
TLG
MBS
TPB
TTA
TTF
TTN
VAB
VCG
VC3
VEF
VCS
VEA
VCB
VHC
VCI
VFS
VIC
VGS
VGC
VIB
VJC
VIX
HVN
VND
VNM
GVR
VRE
VPI
VSC
VOS
VPB
VPG
VTP
VGI
VTO
YEG


In [12]:
# all_df.to_csv('/remote/vast0/share-mv/tien/project/vnstock/notebooks/all_df.csv', index=False)

In [43]:
# all_df = pd.read_csv('/remote/vast0/share-mv/tien/project/vnstock/notebooks/all_df.csv')
# all_df['time'] = pd.to_datetime(all_df['time'])

# all_current_features = []
# for f in os.listdir('/remote/vast0/share-mv/tien/project/vnstock/data/run_results/new_obj_max_10_test_30_retrain_30_top_symbols_score/run_data'):
#     if f.startswith('current_features_'):
#         df = pd.read_csv(f'/remote/vast0/share-mv/tien/project/vnstock/data/run_results/new_obj_max_10_test_30_retrain_30_top_symbols_score/run_data/{f}')
#         df['time'] = f.split('_')[-1].split('.')[0]
#         df['time'] = pd.to_datetime(df['time'])
#         all_current_features.append(df)

# all_current_features = pd.concat(all_current_features).reset_index(drop=True)
# all_cols = all_current_features.columns.tolist()
# merged_df = all_current_features.merge(all_df, on=['symbol', 'time'])

# for c in all_cols:
#     if f'{c}_x' in merged_df.columns and f'{c}_y' in merged_df.columns:
#         diff_res = (merged_df[f'{c}_x'] - merged_df[f'{c}_y']).sum()
#         if diff_res > 1e-5:
#             print(c, diff_res)

In [24]:
# extra_conds = (all_df['close_slope5'].abs() < 0.1) & (all_df['close_slope15'].abs() > 0.1) & (all_df['volume']*all_df['close'] > 5e6)
extra_conds = all_df['volume']*all_df['close'] > 5e6
# all_df['label'] = all_df['label'].apply(lambda x: True if (x == 3 or x == 0) else False)
train_df = all_df[(all_df['time'] < np.datetime64('2024-06-01T00:00:00.000000000')) & extra_conds]
valid_df = all_df[((all_df['time'] >= np.datetime64('2024-06-01T00:00:00.000000000')) & (all_df['time'] < np.datetime64('2024-10-01T00:00:00.000000000'))) & extra_conds]
test_df = all_df[(all_df['time'] >= np.datetime64('2024-10-01T00:00:00.000000000')) & extra_conds]

print(train_df.shape[0], valid_df.shape[0], test_df.shape[0]) 
# for c in train_df.columns:
#     if 'prev' in c:
#         del train_df[c], valid_df[c], test_df[c]

train_df.reset_index(drop=True, inplace=True)
valid_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

train_df_close_cols = train_df['close']
valid_df_close_cols = valid_df['close']
test_df_close_cols = test_df['close']

weight_train = train_df['weight']
weight_valid = valid_df['weight']
weight_test = test_df['weight']

train_df_next_close_cols = train_df['next_close']
valid_df_next_close_cols = valid_df['next_close']
test_df_next_close_cols = test_df['next_close']

train_df_time_cols = train_df['time']
valid_df_time_cols = valid_df['time']
test_df_time_cols = test_df['time']

if 'symbol' in train_df.columns:
    train_df_symbol_cols = train_df['symbol']
    valid_df_symbol_cols = valid_df['symbol']
    test_df_symbol_cols = test_df['symbol']

if 'time' in train_df.columns:
    del train_df['time']
    del valid_df['time']
    del test_df['time']

for c in ['next_close', 'weight']:
    if c in train_df.columns:
        del train_df[c]
        del valid_df[c]
        del test_df[c]

rm_cols = ['open', 'high', 'close', 'low', 'volume', 'label2', 'symbol', 'next_close', 'weight', 'next_min_close']
for c in rm_cols:
    if c in valid_df.columns:
        del test_df[c]
        del train_df[c]
        del valid_df[c]

34769 2181 1379


In [25]:
y_train = train_df['label']
y_valid = valid_df['label']
y_test = test_df['label']
X_train = train_df.drop('label', axis=1)
X_test = test_df.drop('label', axis=1)
X_valid = valid_df.drop('label', axis=1)

print(y_train.value_counts(), y_valid.value_counts(), y_test.value_counts())

def objective(trial):
    dtrain = lgb.Dataset(X_train, label=y_train, weight=weight_train)

    param = {
        "objective": "binary",
        "metric": "binary_logloss",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        # "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        # "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "num_iterations": trial.suggest_int("num_iterations", 20, 1000),
    }
    gbm = lgb.train(param, dtrain)
    preds = gbm.predict(X_valid)
    auc = roc_auc_score(y_valid, preds, sample_weight=weight_valid)
    best_prec = 0
    best_threshold = 0.1
    for t in range(50, 100, 1):
        pred_labels = preds > t/100
        if pred_labels.sum() < 30:
            break
        prec = precision_score(y_valid, pred_labels, sample_weight=weight_valid)
        if prec > best_prec:
            best_threshold = t/100
            best_prec = prec
    total_predicted_positive = (preds > best_threshold).sum()
    print('Best threshold:', best_threshold)
    print('Validation set AUC:', auc)
    print('Validation set Precision:', best_prec)
    print('Number of predicted positive labels:', total_predicted_positive)
    if math.isnan(auc):
        auc = 0
    trial.set_user_attr('model', gbm)
    top_threshold = sorted(preds, reverse=True)[30]
    trial.set_user_attr('best_precision', best_prec)
    trial.set_user_attr('best_threshold', best_threshold)
    trial.set_user_attr('top_threshold', top_threshold)
    return (best_prec + auc) / 2

def callback(study, trial):
    if study.best_trial.number == trial.number:
        study.set_user_attr(key="best_model", value=trial.user_attrs["model"])
        study.set_user_attr(key="best_threshold", value=trial.user_attrs["best_threshold"])
        study.set_user_attr(key="best_precision", value=trial.user_attrs["best_precision"])
        study.set_user_attr(key="top_threshold", value=trial.user_attrs["top_threshold"])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10, callbacks=[callback])

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

best_model = study.user_attrs['best_model']
best_threshold = study.user_attrs['best_threshold']
top_threshold = study.user_attrs['top_threshold']
best_precision = study.user_attrs['best_precision']

[I 2025-03-25 11:27:54,040] A new study created in memory with name: no-name-d5b55697-6b99-43f0-8240-6934bef21bba


label
False    17775
True     16994
Name: count, dtype: int64 label
False    1512
True      669
Name: count, dtype: int64 label
False    989
True     390
Name: count, dtype: int64


Found `num_iterations` in params. Will use it instead of argument


[I 2025-03-25 11:28:27,476] Trial 0 finished with value: 0.6396857135852332 and parameters: {'lambda_l1': 0.014731519261069019, 'lambda_l2': 0.00038343963173966466, 'num_leaves': 157, 'bagging_freq': 7, 'min_child_samples': 84, 'num_iterations': 258}. Best is trial 0 with value: 0.6396857135852332.


Best threshold: 0.91
Validation set AUC: 0.6141324068469649
Validation set Precision: 0.6652390203235016
Number of predicted positive labels: 34


...

Best threshold: 0.93
Validation set AUC: 0.6119394362568188
Validation set Precision: 0.5659423168207999
Number of predicted positive labels: 46


Found `num_iterations` in params. Will use it instead of argument
[I 2025-03-25 11:39:47,557] Trial 8 finished with value: 0.6686464404072514 and parameters: {'lambda_l1': 0.00014657029255094584, 'lambda_l2': 0.002361826769155371, 'num_leaves': 160, 'bagging_freq': 4, 'min_child_samples': 72, 'num_iterations': 935}. Best is trial 8 with value: 0.6686464404072514.


Best threshold: 0.99
Validation set AUC: 0.6167622609744255
Validation set Precision: 0.7205306198400773
Number of predicted positive labels: 37


Found `num_iterations` in params. Will use it instead of argument
[I 2025-03-25 11:40:28,293] Trial 9 finished with value: 0.5896196604187598 and parameters: {'lambda_l1': 0.09295224705331957, 'lambda_l2': 9.178994261208372e-05, 'num_leaves': 63, 'bagging_freq': 6, 'min_child_samples': 72, 'num_iterations': 725}. Best is trial 8 with value: 0.6686464404072514.


Best threshold: 0.92
Validation set AUC: 0.6141190845289228
Validation set Precision: 0.5651202363085968
Number of predicted positive labels: 35
Number of finished trials: 10
Best trial:
  Value: 0.6686464404072514


In [26]:
best_threshold = study.user_attrs['best_threshold']
y_pred_test = best_model.predict(X_test)
auc = roc_auc_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test > best_threshold)
print("AUC: ", auc)
print("Precision: ", prec)
print("Total predictions:", sum(y_pred_test > best_threshold))

AUC:  0.6332399989629515
Precision:  0.7222222222222222
Total predictions: 18


In [66]:
check_test_df = test_df.copy()
check_test_df['label'] = y_test
check_test_df['pred'] = y_pred_test
check_test_df['symbol'] = test_df_symbol_cols
check_test_df['time'] = test_df_time_cols
check_test_df = check_test_df.sort_values(by='pred', ascending=False).reset_index(drop=True)
check_test_df

,close_open_diff,high_low_diff,close_low_diff,volume_level,close_1d_pct_change,close_3d_pct_change,close_5d_pct_change,close_10d_pct_change,close_3d_vs_5d_pct_change,close_5d_vs_10d_pct_change,...,trix,tsi,ttm_histogram,uo,vtx_plus,vtx_minus,label,pred,symbol,time
0,-0.017992,0.038348,0.019666,0,-0.027205,-0.036245,0.009737,0.028770,0.047712,0.018849,...,-42.742004,-1.682024,1.289093,56.790013,1.063636,0.868182,True,0.747783,DRI,2024-09-04
1,0.007843,0.011765,0.007843,1,-0.007722,0.003906,0.079832,0.189815,0.075630,0.101852,...,49.169608,64.940489,1.460418,62.180786,1.371134,0.577320,False,0.724657,TTA,2024-12-13
2,0.000000,0.019417,0.014563,1,0.000000,-0.018779,0.019512,-0.050000,0.039024,-0.068182,...,-40.409390,-15.785981,0.075071,49.671237,0.983871,1.177419,True,0.705018,HAG,2024-09-13
3,-0.004695,0.008255,0.000000,0,-0.003525,-0.010502,0.014354,0.039216,0.025120,0.024510,...,-36.036539,-21.332167,0.058782,46.920076,1.320000,0.792000,True,0.701132,AAA,2024-11-29
4,-0.015873,0.032258,0.000000,0,-0.015873,-0.015873,-0.015873,0.127273,0.000000,0.145455,...,-16.885304,9.123880,0.629893,60.836642,1.269231,0.653846,True,0.684459,AAV,2024-11-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10966,-0.008439,0.011348,0.000000,3,-0.002829,0.002845,0.001420,-0.022191,-0.001420,-0.023578,...,-13.982989,-20.588179,-1.959714,29.835936,0.773333,1.146667,False,0.141291,MSN,2024-12-26
10967,-0.004902,0.012871,0.004950,2,-0.007820,-0.006849,-0.016473,-0.001967,-0.009690,0.014749,...,-7.587139,-18.108708,-0.742821,69.143552,1.047297,0.993243,False,0.137258,VJC,2024-12-03
10968,0.000000,0.023622,0.023622,1,-0.011407,-0.007634,0.000000,0.003861,0.007692,0.003861,...,-13.028424,-13.526130,-2.563250,69.740247,0.937931,0.993103,False,0.136852,DPG,2024-11-15
10969,0.001292,0.001292,0.001292,1,0.001292,0.002587,0.003886,0.011749,0.001295,0.007833,...,0.228026,10.346344,0.143857,73.393054,1.288889,0.844444,False,0.134677,KOS,2024-12-02


In [60]:
for t in check_test_df['time'].unique():
    tdf = check_test_df[check_test_df['time'] == t].sort_values('pred', ascending=False).head(10)
    print(t, precision_score(tdf['label'], tdf['pred'] > 0.), (tdf['pred'] > 0.).sum())

2024-09-04 00:00:00 0.2 10
2024-09-05 00:00:00 0.3 10
2024-09-06 00:00:00 0.2 10
2024-09-09 00:00:00 0.1 10
2024-09-10 00:00:00 0.6 10
2024-09-11 00:00:00 0.4 10
2024-09-12 00:00:00 0.5 10
2024-09-13 00:00:00 0.5 10
2024-09-16 00:00:00 0.8 10
2024-09-17 00:00:00 0.7 10
2024-09-18 00:00:00 0.2 10
2024-09-19 00:00:00 0.3 10
2024-09-20 00:00:00 0.3 10
2024-09-23 00:00:00 0.5 10
2024-09-24 00:00:00 0.2 10
2024-09-25 00:00:00 0.2 10
2024-09-26 00:00:00 0.0 10
2024-09-27 00:00:00 0.0 10
2024-09-30 00:00:00 0.1 10
2024-10-01 00:00:00 0.2 10
2024-10-02 00:00:00 0.0 10
2024-10-03 00:00:00 0.1 10
2024-10-04 00:00:00 0.1 10
2024-10-07 00:00:00 0.3 10
2024-10-08 00:00:00 0.1 10
2024-10-09 00:00:00 0.1 10
2024-10-10 00:00:00 0.3 10
2024-10-11 00:00:00 0.0 10
2024-10-14 00:00:00 0.3 10
2024-10-15 00:00:00 0.3 10
2024-10-16 00:00:00 0.1 10
2024-10-17 00:00:00 0.0 10
2024-10-18 00:00:00 0.1 10
2024-10-21 00:00:00 0.1 10
2024-10-22 00:00:00 0.2 10
2024-10-23 00:00:00 0.1 10
2024-10-24 00:00:00 0.2 10
2

In [50]:
fi = pd.DataFrame([best_model.feature_name(), best_model.feature_importance()]).T
fi.columns = ['name', 'importance']
fi.sort_values(by='importance', ascending=False, inplace=True)
fi.head(20)

,name,importance
27,close_slope15,110
17,cur_close_vs_avg10_diff,106
18,cur_close_vs_avg20_diff,83
72,rsi,62
4,close_1d_pct_change,58
33,close_slope10_vs_slope15_diff,52
80,uo,52
7,close_10d_pct_change,49
32,close_slope5_vs_slope15_diff,42
1,high_low_diff,40


In [ ]:
best_model.feature_importance()

In [17]:
a = pd.DataFrame([test_df_symbol_cols, test_df_close_cols, test_df_next_close_cols, y_pred_test, y_test]).T
a.columns = ['symbol', 'close', 'next_close', 'pred', 'label']
a.sort_values(by='pred', ascending=False).head(20)

,symbol,close,next_close,pred,label
7581,GKM,6.9,7.7,0.923102,True
7575,GKM,8.5,9.3,0.913509,True
7576,GKM,9.3,9.2,0.898364,False
10170,SMC,7.6,7.35,0.869095,False
3578,HTN,9.66,10.65,0.866276,True
10167,SMC,7.61,7.6,0.862833,False
2053,LTG,9.3,9.9,0.861167,True
7583,GKM,5.8,6.9,0.858118,True
7578,GKM,8.3,8.5,0.8575,False
2054,LTG,9.4,9.9,0.857027,True


In [29]:
test_df

,close_open_diff,high_low_diff,close_low_diff,volume_level,close_1d_pct_change,close_3d_pct_change,close_5d_pct_change,close_10d_pct_change,close_3d_vs_5d_pct_change,close_5d_vs_10d_pct_change,...,stoch_rsi_k,stoch_rsi_d,super_trend,trix,tsi,ttm_histogram,uo,vtx_plus,vtx_minus,label
0,0.006122,0.010225,0.008180,3,-0.006048,0.006122,0.008180,0.024948,0.002045,0.016632,...,83.425835,94.475278,23.881072,1.385334,14.321250,0.933518,58.735961,1.142857,0.714286,False
1,-0.006085,0.010225,0.002045,2,-0.006085,-0.006085,0.002045,0.016598,0.008180,0.014523,...,50.092502,77.839446,23.881072,2.368836,14.235273,0.927500,52.628125,1.125000,0.770833,True
2,0.004082,0.006135,0.006135,3,0.004082,-0.008065,0.004082,0.008197,0.012245,0.004098,...,23.807344,52.441894,23.881072,3.264311,14.720037,0.910786,55.599059,1.113636,0.840909,True
3,-0.004073,0.012346,0.006173,3,-0.006098,-0.008114,-0.008114,0.002049,0.000000,0.010246,...,7.048176,26.982674,23.881072,3.992187,13.982668,0.827625,55.888133,0.933333,0.911111,True
4,-0.008180,0.018672,0.006224,3,-0.008180,-0.010204,-0.022177,-0.008180,-0.012097,0.014315,...,7.048176,12.634565,23.881072,4.475730,11.852151,0.723286,50.180422,0.877551,0.918367,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13493,0.029691,0.064457,0.064457,2,0.067077,0.217697,0.292101,0.412052,0.061103,0.092834,...,100.000000,84.955487,14.381444,119.484146,59.856293,3.418221,68.289843,1.364026,0.549251,False
13494,0.015342,0.043944,0.043944,2,0.068627,0.219079,0.390098,0.414504,0.140285,0.017557,...,100.000000,95.618246,15.592800,131.507718,64.064332,3.892829,77.923927,1.405767,0.552008,False
13495,0.033385,0.077216,0.077216,3,0.069077,0.219077,0.391152,0.539239,0.141152,0.106449,...,100.000000,100.000000,16.381520,145.504784,68.119889,4.449393,84.838342,1.453453,0.518519,False
13496,0.044916,0.044916,0.044916,2,0.068652,0.220877,0.392763,0.566987,0.140789,0.125093,...,100.000000,100.000000,17.860368,161.457315,71.897436,5.072957,90.807172,1.549203,0.429241,False


In [20]:
y_pred_test = best_model.predict_proba(X_test)[:, 1]
print("AUC: ", roc_auc_score(y_test, y_pred_test, sample_weight=weight_test))

y_pred_test = y_pred_test > 0.85

cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

print(classification_report(y_test, y_pred_test))

AttributeError: 'Booster' object has no attribute 'predict_proba'

In [13]:
# def objective(trial):
#     dtrain = lgb.Dataset(X_train, label=y_train)

#     param = {
#         "objective": "binary",
#         "metric": "binary_logloss",
#         "verbosity": -1,
#         "boosting_type": "gbdt",
#         # "n_estimators": trial.suggest_int("n_estimators", 50, 200),
#         # "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
#         "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
#         "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
#         "num_leaves": trial.suggest_int("num_leaves", 2, 256),
#         "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
#         "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
#         "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
#         "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
#         # "max_depth": trial.suggest_int("max_depth", 2, 8),
#     }
#     gbm = lgb.train(param, dtrain)
#     preds = gbm.predict(X_valid)
#     auc = roc_auc_score(y_valid, preds)
#     best_prec = 0
#     best_threshold = 0.2
#     for t in range(20, 80, 1):
#         pred_labels = preds > t/100
#         if pred_labels.sum() < 15:
#             break
#         prec = precision_score(y_valid, pred_labels)
#         if prec > best_prec:
#             best_threshold = t/100
#             best_prec = prec
#     print('Best threshold:', best_threshold)
#     trial.set_user_attr('model', gbm)
#     trial.set_user_attr('best_threshold', best_threshold)
#     return auc

# def callback(study, trial):
#     if study.best_trial.number == trial.number:
#         study.set_user_attr(key="best_model", value=trial.user_attrs["model"])
#         study.set_user_attr(key="best_threshold", value=trial.user_attrs["best_threshold"])

# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=100, callbacks=[callback])

# print("Number of finished trials: {}".format(len(study.trials)))

# print("Best trial:")
# trial = study.best_trial

# print("  Value: {}".format(trial.value))

In [14]:
# clf = study.user_attrs['best_model']
# best_threshold = study.user_attrs['best_threshold']

In [15]:
# y_pred_valid = clf.predict(X_valid)
# print("AUC: ", roc_auc_score(y_valid, y_pred_valid))

# y_pred_valid = y_pred_valid > best_threshold

# cm = confusion_matrix(y_valid, y_pred_valid)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm)
# disp.plot()

# print(classification_report(y_valid, y_pred_valid))

In [16]:
# y_pred_test = clf.predict(X_test)
# print("AUC: ", roc_auc_score(y_test, y_pred_test))

# y_pred_test = y_pred_test > best_threshold

# cm = confusion_matrix(y_test, y_pred_test)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm)
# disp.plot()

# print(classification_report(y_test, y_pred_test))

In [17]:
# _y = clf.predict(X_test)
# a = pd.DataFrame([y_test, _y]).T
# a.columns = ['y_test', 'y_pred']
# a.sort_values(by='y_pred', ascending=False).head(10)

In [18]:
# a.sort_values(by='y_pred', ascending=False).head(10)

## Rolling

In [37]:
from datetime import datetime, timedelta
st = datetime(2024, 6, 1)
ed = datetime(2025, 1, 25)

budget = 5
while st < ed:
    print('st:', st)
    train_df = all_df[all_df['time'] < st]
    valid_df = all_df[(all_df['time'] >= st) & (all_df['time'] < st + timedelta(days=14))]
    test_df = all_df[(all_df['time'] >= st + timedelta(days=14)) & (all_df['time'] < st + timedelta(days=21))]

    train_df.reset_index(drop=True, inplace=True)
    valid_df.reset_index(drop=True, inplace=True)
    test_df.reset_index(drop=True, inplace=True)

    train_df_close_cols = train_df['close']
    valid_df_close_cols = valid_df['close']
    test_df_close_cols = test_df['close']

    train_df_next_close_cols = train_df['next_close']
    valid_df_next_close_cols = valid_df['next_close']
    test_df_next_close_cols = test_df['next_close']

    if 'symbol' in train_df.columns:
        train_df_symbol_cols = train_df['symbol']
        valid_df_symbol_cols = valid_df['symbol']
        test_df_symbol_cols = test_df['symbol']

    if 'next_close' in train_df.columns:
        del train_df['next_close']
        del valid_df['next_close']
        del test_df['next_close']

    # train_df_label2_cols = train_df['label2']
    # valid_df_label2_cols = valid_df['label2']
    # test_df_label2_cols = test_df['label2']

    rm_cols = ['open', 'high', 'close', 'low', 'volume', 'time', 'label2', 'symbol', 'next_close']
    for c in rm_cols:
        if c in valid_df.columns:
            del test_df[c]
            del train_df[c]
            del valid_df[c]

    print("Loading data...")

    y_train = train_df['label']
    y_valid = valid_df['label']
    y_test = test_df['label']
    X_train = train_df.drop('label', axis=1)
    X_valid = valid_df.drop('label', axis=1)
    X_test = test_df.drop('label', axis=1)

    def objective(trial):
        dtrain = lgb.Dataset(X_train, label=y_train)

        param = {
            "objective": "binary",
            "metric": "binary_logloss",
            "verbosity": -1,
            "boosting_type": "gbdt",
            # "n_estimators": trial.suggest_int("n_estimators", 50, 200),
            # "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 2, 256),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            # "max_depth": trial.suggest_int("max_depth", 2, 8),
        }
        gbm = lgb.train(param, dtrain)
        preds = gbm.predict(X_valid)
        auc = roc_auc_score(y_valid, preds)
        best_prec = 0
        best_threshold = 0.2
        for t in range(20, 80, 1):
            pred_labels = preds > t/100
            if pred_labels.sum() < 10:
                break
            prec = precision_score(y_valid, pred_labels)
            if prec > best_prec:
                best_threshold = t/100
                best_prec = prec
        print('Best threshold:', best_threshold)
        trial.set_user_attr('model', gbm)
        trial.set_user_attr('best_threshold', best_threshold)
        return (best_prec + auc) / 2

    def callback(study, trial):
        if study.best_trial.number == trial.number:
            study.set_user_attr(key="best_model", value=trial.user_attrs["model"])
            study.set_user_attr(key="best_threshold", value=trial.user_attrs["best_threshold"])

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50, callbacks=[callback])

    print("Number of finished trials: {}".format(len(study.trials)))

    print("Best trial:")
    trial = study.best_trial

    print("  Value: {}".format(trial.value))

    clf = study.user_attrs['best_model']
    best_threshold = study.user_attrs['best_threshold']
    if X_test.shape[0] == 0:
        st = st + timedelta(days=7)
        continue

    y_pred_test = clf.predict(X_test)

    print("AUC: ", roc_auc_score(y_test, y_pred_test))

    # y_pred_test = y_pred_test > best_threshold

    # cm = confusion_matrix(y_test, y_pred_test)
    # disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    # disp.plot()

    # print(classification_report(y_test, y_pred_test))

    test_df['label'] = y_test
    test_df['label_pred'] = y_pred_test
    test_df['close'] = test_df_close_cols
    test_df['next_close'] = test_df_next_close_cols
    test_df['symbol'] = test_df_symbol_cols

    top5 = test_df.sort_values(by='label_pred', ascending=False).head(5)
    top5 = top5[top5['label_pred'] > best_threshold]
    new_budget = 0
    for i, row in top5.iterrows():
        print(row['symbol'], row['next_close'], row['close'])
        if not np.isnan(row['next_close']):
            new_budget += (row['next_close'] / row['close']-0.002) * (budget / top5.shape[0])
        else:
            new_budget += budget / top5.shape[0]

    if top5.shape[0] == 0:
        new_budget = budget

    budget = new_budget
    st = st + timedelta(days=7)

    print('Current budget:', budget)

st: 2024-06-01 00:00:00


[I 2025-02-08 00:32:54,775] A new study created in memory with name: no-name-7d96a4ca-7a68-4a78-99b7-272ce6b74fe5


Loading data...


[I 2025-02-08 00:32:59,103] Trial 0 finished with value: 0.912547934813224 and parameters: {'lambda_l1': 2.058540259643051e-07, 'lambda_l2': 1.9050111793822312e-08, 'num_leaves': 31, 'feature_fraction': 0.8568553284311652, 'bagging_fraction': 0.7011239974734746, 'bagging_freq': 7, 'min_child_samples': 14}. Best is trial 0 with value: 0.912547934813224.


Best threshold: 0.79


[W 2025-02-08 00:33:00,454] Trial 1 failed with parameters: {'lambda_l1': 1.305033008283587e-06, 'lambda_l2': 0.13181261566808528, 'num_leaves': 196, 'feature_fraction': 0.6836232781674018, 'bagging_fraction': 0.536606035521579, 'bagging_freq': 6, 'min_child_samples': 7} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/data0/tien/anaconda3/envs/vns/lib/python3.11/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_2360688/2835710835.py", line 73, in objective
    gbm = lgb.train(param, dtrain)
          ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data0/tien/anaconda3/envs/vns/lib/python3.11/site-packages/lightgbm/engine.py", line 307, in train
    booster.update(fobj=fobj)
  File "/data0/tien/anaconda3/envs/vns/lib/python3.11/site-packages/lightgbm/basic.py", line 4136, in update
    _LIB.LGBM_BoosterUpdateOneIter(
KeyboardInterrupt
[W 2025-02-0

KeyboardInterrupt: 

In [46]:
print(train_df.columns.tolist())

['time', 'open', 'high', 'low', 'close', 'volume', 'close_open_diff', 'high_low_diff', 'close_low_diff', 'volume_level', 'close_1d_pct_change', 'close_3d_pct_change', 'close_5d_pct_change', 'close_10d_pct_change', 'close_3d_vs_5d_pct_change', 'close_5d_vs_10d_pct_change', 'volume_1d_pct_change', 'volume_3d_pct_change', 'volume_5d_pct_change', 'volume_10d_pct_change', 'volume_3d_vs_5d_pct_change', 'volume_5d_vs_10d_pct_change', 'cur_close_vs_avg5_diff', 'cur_close_vs_avg10_diff', 'cur_close_vs_avg20_diff', 'cur_volume_vs_avg5_diff', 'cur_volume_vs_avg10_diff', 'cur_volume_vs_avg20_diff', 'volume_slope5', 'volume_slope10', 'volume_slope15', 'close_slope5', 'close_slope10', 'close_slope15', 'volume_slope5_vs_slope10_diff', 'volume_slope5_vs_slope15_diff', 'volume_slope10_vs_slope15_diff', 'close_slope5_vs_slope10_diff', 'close_slope5_vs_slope15_diff', 'close_slope10_vs_slope15_diff', 'adx_plus_di', 'adx_minus_di', 'ao', 'aroon_up', 'aroon_down', 'atr', 'bop', 'chop', 'coppock_curve', 'dpo

In [55]:
conditions = (train_df['close_slope5'] > 0) & (train_df['volume_slope10'] > 0.1)
train_df[conditions]['label'].value_counts()

label
False    36034
True     11075
Name: count, dtype: int64